In [1]:
import sys
sys.path.append("..")
from src.data import load_isot

df = load_isot()
print(df.shape)
print(df.columns.tolist())

(44898, 6)
['title', 'text', 'subject', 'date', 'label', 'date_raw']


In [4]:
print(df["label"].value_counts())
print("missing values:\n", df.isna().sum())
print("duplicate rows:", df.duplicated(subset=["title","text"]).sum())

label
1    23481
0    21417
Name: count, dtype: int64
missing values:
 title          0
text           0
subject        0
date       33030
label          0
dtype: int64
duplicate rows: 5793


In [5]:
print(df.groupby("label")["date"].agg(["min","max"]))
print("dates that failed to parse:",df["date"].isna().sum())

             min        max
label                      
0            NaT        NaT
1     2015-05-01 2017-12-31
dates that failed to parse: 33030


In [8]:
df["words"] = df["text"].str.split().str.len()
print(df.groupby("label")["words"].describe())

         count        mean         std  min    25%    50%    75%     max
label                                                                   
0      21417.0  385.640099  274.006204  0.0  148.0  359.0  525.0  5172.0
1      23481.0  423.197905  408.388890  0.0  240.0  363.0  506.0  8135.0


In [7]:
for lbl in [0,1]:
    print("=== label",lbl, "===")
    for t in df[df["label"] == lbl]["text"].head(5):
        print(t[:200])
        print("---")

=== label 0 ===
WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal 
---
WASHINGTON (Reuters) - Transgender people will be allowed for the first time to enlist in the U.S. military starting on Monday as ordered by federal courts, the Pentagon said on Friday, after Presiden
---
WASHINGTON (Reuters) - The special counsel investigation of links between Russia and President Trump’s 2016 election campaign should continue without interference in 2018, despite calls from some Trum
---
WASHINGTON (Reuters) - Trump campaign adviser George Papadopoulos told an Australian diplomat in May 2016 that Russia had political dirt on Democratic presidential candidate Hillary Clinton, the New Y
---
SEATTLE/WASHINGTON (Reuters) - President Donald Trump called on the U.S. Postal Service on Friday to charge “much more” to ship packages for Amazon (AMZN.O), pickin

In [2]:
bad = df[df["date"].isna()]
print(bad.groupby("label")["date_raw"].apply(lambda s: s.head(5).tolist()))
good = df[df["date"].notna()]
print(good.groupby("label")["date_raw"].apply(lambda s: s.head(5).tolist()))

label
0    [December 31, 2017 , December 29, 2017 , Decem...
1    [19-Feb-18, 19-Feb-18, 19-Feb-18, 18-Feb-18, 1...
Name: date_raw, dtype: object
label
1    [December 31, 2017, December 31, 2017, Decembe...
Name: date_raw, dtype: object


In [3]:
print(df[df.duplicated(subset=["title", "text"])].groupby("label").size())
both = df.groupby(["title", "text"])["label"].nunique()
print("texts that appear with BOTH labels:", (both > 1).sum())

label
0     220
1    5573
dtype: int64
texts that appear with BOTH labels: 0


In [5]:
import pandas as pd
print(pd.crosstab(df["subject"], df["label"]))

label                0     1
subject                     
Government News      0  1570
Middle-east          0   778
News                 0  9050
US_News              0   783
left-news            0  4459
politics             0  6841
politicsNews     11272     0
worldnews        10145     0


In [6]:
has_reuters = df["text"].str.contains(r"\(Reuters\)", regex=True)
print(has_reuters.groupby(df["label"]).mean())

label
0    0.992062
1    0.000383
Name: text, dtype: float64


In [9]:
print((df["words"] < 10).groupby(df["label"]).sum())
print(df[df["words"] < 10][["label", "title", "text"]].head(10))

label
0      1
1    866
Name: words, dtype: int64
       label                                              title  \
9358       1  https://100percentfedup.com/served-roy-moore-v...   
10923      1  TAKE OUR POLL: Who Do You Think President Trum...   
11041      1  Joe Scarborough BERATES Mika Brzezinski Over “...   
11117      1  MY FAVORITE EXCUSES…Featuring Hillary Rotten C...   
11190      1  WATCH TUCKER CARLSON Scorch Sanctuary City May...   
11225      1  MAYOR OF SANCTUARY CITY: Trump Trying To Make ...   
11236      1  SHOCKER: Public School Turns Computer Lab Into...   
11241      1  BOOM! SEAN SPICER: “Trump Sold Hotels In Russi...   
11247      1  MICHAEL FLYNN’S LAWYER Releases Statement Scor...   
11249      1  ROB SCHNEIDER Nails The Russia Conspiracy Theo...   

                                                    text  
9358   https://100percentfedup.com/served-roy-moore-v...  
10923                                                     
11041                              

In [10]:
curly = df["text"].str.contains("[’“”]", regex=True)
print(curly.groupby(df["label"]).mean())

label
0    0.505066
1    0.000000
Name: text, dtype: float64


In [11]:
df["date"] = pd.to_datetime(df["date_raw"].str.strip(), format="mixed", errors="coerce")
print(df["date"].isna().groupby(df["label"]).sum())
print(df.groupby("label")["date"].agg(["min", "max"]))

label
0     0
1    10
Name: date, dtype: int64
             min        max
label                      
0     2016-01-13 2017-12-31
1     2015-03-31 2018-02-19


In [12]:
from sklearn.metrics import f1_score
rule_pred = (~has_reuters).astype(int)   # predict fake when no "(Reuters)" tag
print("one-word rule, macro-F1:", f1_score(df["label"], rule_pred, average="macro"))

one-word rule, macro-F1: 0.996003370224352


In [13]:
from src.clean import clean_isot
clean = clean_isot(df)
print(len(df), "->", len(clean))
print(clean["label"].value_counts())

44898 -> 38494
label
0    21196
1    17298
Name: count, dtype: int64
